# KD teacher-student: BEATs (nhiễu sạch) → FRCRN + MobileViT-S, học bằng q/k/v và CRD

| Nhánh | Đường đi | Trạng thái |
| --- | --- | --- |
| Teacher | **nhiễu thật** → chuẩn hoá RMS → fbank → BEATs fine-tune (`pretrained/beat-ft`, fold 0) | đóng băng |
| Student | mixture → FRCRN (`pretrained/FRCRN/best_e037_…pt`, mask) → n̂ → log-mel → **MobileViT 12 lớp** (L = 2, 6, 4; d = 768; 12 head) → attention pooling → logit 36 | FRCRN đóng băng, MobileViT huấn luyện |
| KD | **q/k/v** của 12 lớp self-attention BEATs ← 12 lớp Transformer của student (khối 1: lớp 1-2 ↔ BEATs 1-2, khối 2: lớp 1-6 ↔ BEATs 3-8, khối 3: lớp 1-4 ↔ BEATs 9-12); **CRD** trên embedding đã pool; **logit** 36 lớp của head teacher | chỉ lúc train |

**Loss** = CE (label smoothing 0.1) + λ_qkv · L_qkv + λ_crd · L_crd + λ_logit · L_logit, các λ tăng tuyến tính trong `warmup_epochs` epoch đầu.

- **L_qkv** (mỗi cặp lớp, mỗi loại q, k, v): `1 − cos` theo từng token + `REL_WEIGHT` · KL giữa ma trận quan hệ `softmax(xxᵀ/√d)` theo từng head (kiểu MiniLM). Cosine không phụ thuộc thang đo, nên q, k, v có độ lớn khác nhau vẫn so sánh được. Student có cùng d = 768 và 12 head với BEATs nên so q, k, v trực tiếp, không cần adapter. Student lấy q/k/v từ `attn.qkv` của từng lớp (trung bình theo 4 vị trí trong patch); q/k/v của teacher được pool trung bình về lưới token của khối student (8×25, 4×13, 2×7 với clip 4 s).
- **L_crd** (Tian et al., 2020): hai head chiếu về 128 chiều, L2-norm. Positive là cùng mẫu ở hai view, negative lấy `CRD_NEGATIVES` mẫu ngẫu nhiên từ memory bank (cập nhật momentum). Negative cùng lớp với anchor bị loại, để tránh false negative vì tập chỉ có 36 lớp.
- **L_logit** (Hinton et al., 2015): `T² · KL(softmax(z_t/T) ‖ softmax(z_s/T))` với T = 4. Teacher nhìn nhiễu sạch nên logit của nó là nhãn mềm "có đặc quyền"; đổi lại teacher fold 0 chỉ đạt khoảng 0.74 accuracy, nên nhãn mềm có nhiễu và CE vẫn giữ vai trò neo.
- **Đánh giá:** 9 chỉ số (top1, top3, balanced acc, precision, recall, macro-F1, micro-F1, mAP, macro-AUC) tại từng mức SNR (−5…20) và trung bình. Checkpoint chọn theo `macro_acc` (balanced acc trung bình theo SNR), đúng chuẩn repo.

`RUN_TRAINING = False` chỉ chạy smoke test nhỏ (64 mẫu, 1 epoch) vào `output/test/`. Cell (1) có cờ `USE_QKV`, `USE_CRD`, `USE_LOGIT` để ablation: CE → +logit → +qkv → +crd → +cả ba. `USE_NHAT_CACHE` tính trước n̂ của FRCRN một lần để epoch nhanh hơn nhiều.

<p align="center"></p>

<p align="center"><em>Hình 1. Tổng quan: teacher nhận nhiễu thật, student nhận mixture, hai loss KD nối hai nhánh.</em></p>

In [ ]:
# (0) Import
import copy
import math
import sys
from collections import namedtuple
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.optim import AdamW
from torch.utils.data import Dataset, Subset
from tqdm.auto import tqdm

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# (1) Cấu hình thí nghiệm. Đổi MỘT cờ mỗi lần để ablation.
EXPERIMENT = "beat-mv-kd"
RUN_TRAINING = False          # False: smoke test nhỏ vào output/test
USE_QKV = True                # L_qkv
USE_CRD = True                # L_crd
USE_LOGIT = True              # L_logit: KL(softmax(z_t/T) || softmax(z_s/T)) trên logit 36 lớp của teacher fold 0 (T = 4, trọng số 1,0 trong KD); False để tắt
USE_DRIVE = True              # backup best.pt, resume.pt, history, log lên Google Drive (đăng nhập ở cell 2b)
DRIVE_DIR = "FA26_Capstone/Truong/beat-mv-kd-v3"  # thư mục mới: beat-mv-kd và beat-mv-kd-v2 trên Drive đã có bản trùng tên (MultipleFilesError)
USE_NHAT_CACHE = True         # tính trước n^ = FRCRN(mixture) một lần (FRCRN đóng băng, mixture không augment nên n^ không đổi); chỉ dùng khi RUN_TRAINING
NHAT_CACHE_DIR = ROOT / "cache" / "nhat"
NHAT_ZIP = ROOT / "nhat_cache.zip"                     # nén lossless thư mục cache/ (khoảng 8,6 GB), nằm NGOÀI cache/ để không tự nén chính nó
NHAT_DRIVE_ZIP = f"{DRIVE_DIR}/nhat/nhat_cache.zip"    # bản lưu trên Drive: sandbox mới chỉ cần tải về và giải nén thay vì tính lại (khoảng 17 phút)
UPLOAD_NHAT_TO_DRIVE = True                            # đẩy zip lên Drive một lần nếu Drive chưa có (khoảng 8 phút, chặn huấn luyện); False để bỏ qua
USE_AUG = True                # augmentation nhất quán trên cặp (n^ cho student, nhiễu thật cho teacher): dịch vòng thời gian + mixup; cần USE_NHAT_CACHE
AUG = dict(shift=True, mixup_prob=0.5, mixup_alpha=0.4)  # mixup_prob: xác suất một batch được trộn; alpha của Beta(alpha, alpha)
NUM_WORKERS = 8

BEAT_FOLD = 0
BEAT_FT_PATH = ROOT / "pretrained" / "beat-ft" / f"beats_teacher_fold{BEAT_FOLD}_weights.pt"
FRCRN_PATH = ROOT / "pretrained" / "FRCRN" / "best_e037_20261020T203958Z.pt"
BEATS_CONFIG = ROOT / "config" / "beat.yaml"
FRCRN_CONFIG = ROOT / "config" / "frcrn.yaml"

TEACHER_LAYERS = tuple(range(1, 13))  # cả 12 lớp BEATs (đếm từ 1) lấy q/k/v
# MobileViT 12 lớp: khối 1 (2 lớp) <-> BEATs 1-2, khối 2 (6 lớp) <-> BEATs 3-8, khối 3 (4 lớp) <-> BEATs 9-12
STUDENT = dict(dim=768, heads=12, ffn_dim=768, layers=(2, 6, 4), channels=(96, 128, 160), patch=2)
assert sum(STUDENT["layers"]) == len(TEACHER_LAYERS) and STUDENT["dim"] == 768 and STUDENT["heads"] == 12
TARGET_RMS = 0.07925          # chuẩn hoá RMS nhiễu trước fbank, như reviewkd.yaml
NUM_CLASSES = 36
SNR_LEVELS = (-5, 0, 5, 10, 15, 20)

KD = dict(
    qkv_weight=1.0,           # λ_qkv (chưa tinh chỉnh)
    rel_weight=1.0,           # trọng số KL quan hệ so với cosine bên trong L_qkv
    crd_weight=0.5,           # λ_crd (chưa tinh chỉnh)
    logit_weight=1.0,         # λ_logit (chưa tinh chỉnh)
    logit_temperature=4.0,    # như loss.kd.temperature trong beat.yaml
    crd_dim=128,
    crd_negatives=4096,
    crd_temperature=0.07,
    crd_momentum=0.5,
    warmup_epochs=5,
)

FRONTEND = dict(sample_rate=16000, n_fft=512, win_length=400, hop_length=160, n_mels=128,
                f_min=20.0, f_max=8000.0, num_frames=398, log_eps=1e-6, norm_eps=1e-5)
SPECAUG = dict(freq_masks=2, freq_width=16, time_masks=2, time_width=24)

TRAIN = dict(lr=1e-4, weight_decay=0.1, betas=(0.9, 0.999), warmup_ratio=0.05,
             lr_factor=0.5, lr_patience=2, lr_threshold=0.005, min_lr=1e-6,  # ReduceLROnPlateau theo val macro_acc, mỗi epoch
             label_smoothing=0.1, grad_clip=1.0, batch_size=32, eval_batch_size=64,
             epochs=50, patience=15, dropout=0.3, drop_path=0.1, seed=42)

In [ ]:
# (2) Đường dẫn đầu ra, logger, thiết bị
from src.utils.logger import setup_logger
from src.utils.paths import RunPaths
from src.utils.seed import set_seed

output_dir = Path("output") if RUN_TRAINING else Path("output") / "test"
run = RunPaths.new(EXPERIMENT if RUN_TRAINING else f"{EXPERIMENT}-smoke", ROOT / output_dir)
run.create()
logger = setup_logger(EXPERIMENT, run.log_file)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp = lambda: torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda")
logger.info("run %s | device %s | RUN_TRAINING=%s | qkv=%s crd=%s logit=%s", run.run_id, device, RUN_TRAINING, USE_QKV, USE_CRD, USE_LOGIT)

In [ ]:
# (2b) Google Drive: lần đầu hiện liên kết đăng nhập, đăng nhập xong cell mới chạy tiếp. fs = None khi tắt hoặc smoke test.
fs = None
if USE_DRIVE and RUN_TRAINING:
    from gdrive_fsspec import GoogleDriveFileSystem

    fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False})
logger.info("drive: %s", f"bật, thư mục {DRIVE_DIR}" if fs is not None else "tắt")

## Teacher: nhiễu → BEATs (fold 0, đóng băng)

Checkpoint `beat-ft` lưu theo tên khoá BEATs gốc (`encoder.model.*`, `head.*`), nên cần đổi tên sang `BEATClassifier`. `convert_checkpoint` của repo lo phần `backbone.*`, `fc1/fc2` và `patch_embedding`; head đổi tay thành `head.fc.*`. Nạp xong kiểm tra không thiếu hay thừa khoá nào.

q/k/v lấy bằng forward hook trên `q_proj/k_proj/v_proj` của `GatedRelPosAttention`, tức là trước khi chia head. Mỗi tensor có dạng (B, N, 768) với N = 8 × 24 token, thứ tự (thời gian, tần số), cùng thứ tự với `tokens_to_grid` của repo.

<p align="center"></p>

<p align="center"><em>Hình 2. Teacher: lấy q/k/v của cả 12 lớp và embedding cho CRD.</em></p>

In [ ]:
# (3) Teacher
from src.model.beat import BEATClassifier
from src.model.reviewkd import rms_normalize

TeacherOutput = namedtuple("TeacherOutput", "qkv embedding logits")


def load_fold_teacher(path: Path) -> BEATClassifier:
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    state = checkpoint["state_dict"]
    backbone = {k.removeprefix("encoder.model."): v for k, v in state.items() if k.startswith("encoder.model.")}
    beats = BEATClassifier.from_yaml(BEATS_CONFIG)
    converted = beats.convert_checkpoint({"model": backbone})
    converted["head.fc.weight"], converted["head.fc.bias"] = state["head.weight"], state["head.bias"]
    result = beats.load_state_dict(converted, strict=False)
    if result.missing_keys or result.unexpected_keys:
        raise ValueError(f"missing {result.missing_keys[:5]}, unexpected {result.unexpected_keys[:5]}")
    logger.info("teacher fold %s | epoch %s | clean_val_accuracy %.4f | macro_f1 %.4f", checkpoint["fold"],
                checkpoint["best_epoch"], checkpoint["clean_val_accuracy"], checkpoint["clean_val_macro_f1"])
    return beats


class QKVTeacher(nn.Module):
    """nhiễu thật -> RMS -> fbank -> BEATs. Trả q/k/v (L, 3, B, N, 768), embedding (B, 768), logits (B, 36)."""

    def __init__(self, beats: BEATClassifier, layers: tuple[int, ...], target_rms: float) -> None:
        super().__init__()
        self.beats = beats.freeze()
        self.layers = tuple(layers)
        self.target_rms = target_rms
        self.freq_bins = beats.frontend.num_mel_bins // beats.config.input_patch_size
        self._captured: dict[tuple[int, str], Tensor] = {}
        for layer in self.layers:
            attention = beats.backbone.encoder.layers[layer - 1].self_attn
            for name in "qkv":
                getattr(attention, f"{name}_proj").register_forward_hook(self._hook(layer, name))

    def _hook(self, layer: int, name: str):
        def capture(module, inputs, output):
            self._captured[(layer, name)] = output
        return capture

    def train(self, mode: bool = True) -> "QKVTeacher":
        return super().train(False)

    @torch.no_grad()
    def forward(self, noise: Tensor) -> TeacherOutput:
        fbank = self.beats.frontend(rms_normalize(noise.float(), self.target_rms))  # fp32: kaldi fbank
        with amp():
            output = self.beats(fbank, is_waveform=False)
        qkv = torch.stack([torch.stack([self._captured[(layer, name)].float() for name in "qkv"]) for layer in self.layers])
        self._captured.clear()
        return TeacherOutput(qkv, output.features.float().mean(dim=1), output.logits.float())


teacher = QKVTeacher(load_fold_teacher(BEAT_FT_PATH), TEACHER_LAYERS, TARGET_RMS).to(device)
with torch.no_grad():
    _probe = teacher(torch.randn(2, 4 * FRONTEND["sample_rate"], device=device) * 0.05)
TEACHER_TOKENS = _probe.qkv.shape[3]
GRID = (teacher.freq_bins, TEACHER_TOKENS // teacher.freq_bins)  # (tần số, thời gian) = (8, 24)
logger.info("teacher qkv %s | embedding %s | lưới token %s", tuple(_probe.qkv.shape), tuple(_probe.embedding.shape), GRID)
del _probe

## Student: mixture → FRCRN (mask, đóng băng) → n̂ → log-mel → MobileViT 12 lớp

`FRCRNSeparator.forward` đã trả n̂ = mixture − ŝ khi `target: speech` trong `frcrn.yaml`. Student là MobileViT-S sửa theo đặc tả: stem, các khối MV2 và ba khối MobileViT với **L = (2, 6, 4) lớp Transformer, d = 768, 12 head × 64, FFN 768, patch 2×2**, conv 1×1 cuối 768 kênh. Tổng 12 lớp, mỗi lớp khớp một lớp BEATs về d và số head nên không cần adapter: q, k, v lấy thẳng từ `attn.qkv` của từng lớp (trung bình theo 4 vị trí trong patch).

Đầu ra: trung bình theo trục tần số, rồi attention pooling theo thời gian cho từng lớp (nhánh `att` softmax, nhánh `cla` logit). Dự án này là phân loại **đơn nhãn 36 lớp, clip 4 s (398 khung)**, nên logit đi vào CE; đặc tả gốc là AudioSet 527 lớp đa nhãn, 10 s (998 khung) với BCE và sigmoid, và những phần đó không áp dụng ở đây. FRCRN và log-mel chạy fp32, chỉ MobileViT chạy bf16.

<p align="center"></p>

<p align="center"><em>Hình 3. Student: FRCRN cho n̂, MobileViT 12 lớp (2 + 6 + 4) và điểm lấy q/k/v của từng lớp.</em></p>

In [ ]:
# (4) Student: MobileViT 12 lớp, q/k/v của từng lớp lấy bằng hook trên attn.qkv
from timm.models.byobnet import ByobNet, ByoBlockCfg, ByoModelCfg
from timm.models.mobilevit import MobileVitBlock

from src.model.frcrn import FRCRNSeparator
from src.model.kdmoe.frontend import LogMel, SpecAugment

StudentOutput = namedtuple("StudentOutput", "logits qkv grids embedding")


def mobilevit12_cfg(spec: dict) -> ByoModelCfg:
    def inverted(d, c, s):
        return ByoBlockCfg(type="bottle", d=d, c=c, s=s, gs=1, br=4.0, block_kwargs=dict(bottle_in=True, linear_out=True))

    def vit_stage(c, depth):  # MV2 stride 2 rồi khối MobileViT
        return (inverted(1, c, 2), ByoBlockCfg(type="mobilevit", d=1, c=c, s=1, block_kwargs=dict(
            transformer_dim=spec["dim"], transformer_depth=depth, patch_size=spec["patch"], num_heads=spec["heads"],
            mlp_ratio=spec["ffn_dim"] / spec["dim"])))

    blocks = (inverted(1, 32, 1), inverted(3, 64, 2), *[vit_stage(c, d) for c, d in zip(spec["channels"], spec["layers"])])
    return ByoModelCfg(blocks=blocks, stem_chs=16, stem_type="3x3", stem_pool="", downsample="", act_layer="silu",
                       num_features=spec["dim"])


class FRCRNMobileViT(nn.Module):
    def __init__(self, separator: FRCRNSeparator, spec: dict) -> None:
        super().__init__()
        self.separator = separator.freeze()
        self.frontend = LogMel(**FRONTEND)
        self.spec_augment = SpecAugment(**SPECAUG)
        self.net = ByobNet(mobilevit12_cfg(spec), num_classes=0, in_chans=1, drop_path_rate=TRAIN["drop_path"])
        self.dropout = nn.Dropout(TRAIN["dropout"])
        self.att = nn.Linear(spec["dim"], NUM_CLASSES)  # attention pooling: softmax theo thời gian, riêng từng lớp
        self.cla = nn.Linear(spec["dim"], NUM_CLASSES)
        self._qkv, self._grids, self._grid, self._batch = [], [], None, 0
        for block in (m for m in self.net.modules() if isinstance(m, MobileVitBlock)):
            block.conv_1x1.register_forward_hook(self._record_grid(spec["patch"]))
            for layer in block.transformer:
                layer.attn.qkv.register_forward_hook(self._record_qkv)

    def _record_grid(self, patch: int):
        def hook(module, inputs, output):
            self._grid = (math.ceil(output.shape[-2] / patch), math.ceil(output.shape[-1] / patch))  # (tần số, thời gian)
        return hook

    def _record_qkv(self, module, inputs, output) -> None:
        # output (B*P, N, 3D) -> q, k, v (B, N, D), trung bình theo P vị trí trong patch; N theo thứ tự (tần số, thời gian)
        q, k, v = (t.view(self._batch, -1, t.shape[1], t.shape[2]).float().mean(dim=1) for t in output.chunk(3, dim=-1))
        self._qkv.append(torch.stack([q, k, v]))
        self._grids.append(self._grid)

    def train(self, mode: bool = True) -> "FRCRNMobileViT":
        super().train(mode)
        self.separator.eval()
        return self

    def forward(self, waveform: Tensor | None = None, noise_hat: Tensor | None = None) -> StudentOutput:
        source = noise_hat if noise_hat is not None else waveform
        with torch.no_grad(), torch.autocast(source.device.type, enabled=False):
            if noise_hat is None:  # không có cache: chạy FRCRN trực tiếp
                noise_hat = self.separator(waveform.float())
            x, _ = self.frontend(noise_hat.float())
        x = self.spec_augment(x)
        self._qkv, self._grids, self._batch = [], [], x.shape[0]
        with amp():
            h = self.net.forward_features(x)                      # (B, 768, H, W)
            z = self.dropout(h.mean(dim=2).transpose(1, 2))      # trung bình theo tần số -> (B, T, 768)
            weight = self.att(z).float().softmax(dim=1)          # softmax theo thời gian
            logits = (weight * self.cla(z).float()).sum(dim=1)   # (B, 36)
        return StudentOutput(logits, tuple(self._qkv), tuple(self._grids), h.float().mean(dim=(2, 3)))

    def predict_proba(self, waveform: Tensor | None = None, noise_hat: Tensor | None = None) -> Tensor:
        return self(waveform, noise_hat).logits.softmax(dim=-1)


def build_student() -> FRCRNMobileViT:
    separator = FRCRNSeparator.from_yaml(FRCRN_CONFIG)
    report = separator.load_pretrained(FRCRN_PATH)
    logger.info("FRCRN %s | missing %d unexpected %d", FRCRN_PATH.name, len(report.missing), len(report.unexpected))
    return FRCRNMobileViT(separator, STUDENT)


student = build_student().to(device)
with torch.no_grad():
    _probe = student.eval()(torch.randn(2, 4 * FRONTEND["sample_rate"], device=device) * 0.05)
EMBED_CHANNELS = _probe.embedding.shape[1]
assert len(_probe.qkv) == len(TEACHER_LAYERS), f"student có {len(_probe.qkv)} lớp, teacher {len(TEACHER_LAYERS)}"
logger.info("student %.2fM tham số (transformer %.2fM) | embedding %d", sum(p.numel() for p in student.net.parameters()) / 1e6,
            sum(p.numel() for n, p in student.net.named_parameters() if ".transformer." in n) / 1e6, EMBED_CHANNELS)
_owner = [block for block, depth in enumerate(STUDENT["layers"], 1) for _ in range(depth)]
for _index, (_qkv, _grid) in enumerate(zip(_probe.qkv, _probe.grids)):
    logger.info("student lớp %2d (khối %d) q/k/v %s lưới token %s <-> BEATs lớp %2d (lưới %s)", _index + 1, _owner[_index],
                tuple(_qkv.shape), _grid, TEACHER_LAYERS[_index], GRID)
del _probe

## Loss KD: q/k/v và CRD

<p align="center"></p>

<p align="center"><em>Hình 4. q/k/v của từng lớp student so với lớp BEATs tương ứng bằng cosine và KL quan hệ (không cần adapter).</em></p>

<p align="center"></p>

<p align="center"><em>Hình 5. CRD: hai head chiếu, memory bank hai view và InfoNCE đối xứng.</em></p>

In [ ]:
# (5) Pool q/k/v teacher về lưới student, loss q/k/v, loss CRD
def pool_teacher(qkv: Tensor, grid_teacher: tuple[int, int], grid_student: tuple[int, int]) -> Tensor:
    """(3, B, N_t, D), token BEATs theo thứ tự (thời gian, tần số) -> (3, B, N_s, D) trên lưới student, thứ tự (tần số, thời gian)."""
    three, batch, tokens, dim = qkv.shape
    freq, time = grid_teacher
    x = qkv.reshape(three * batch, time, freq, dim).permute(0, 3, 2, 1)  # (3B, D, F, T)
    if (freq, time) != tuple(grid_student):
        x = F.adaptive_avg_pool2d(x, grid_student)
    return x.permute(0, 2, 3, 1).reshape(three, batch, -1, dim)


def relation(x: Tensor, heads: int) -> Tensor:
    """(B, N, D) -> softmax(x_h x_h^T / sqrt(d)) theo từng head, (B*H*N, N)."""
    batch, tokens, dim = x.shape
    x = x.view(batch, tokens, heads, dim // heads).transpose(1, 2)
    return (x @ x.transpose(-2, -1) / math.sqrt(dim // heads)).reshape(-1, tokens)


class QKVLoss(nn.Module):
    def __init__(self, heads: int = 12, rel_weight: float = 1.0) -> None:
        super().__init__()
        self.heads, self.rel_weight = heads, rel_weight

    def forward(self, student_qkv: Tensor, teacher_qkv: Tensor) -> tuple[Tensor, Tensor]:
        """Cả hai (3, B, N, D). Trả (cosine, KL quan hệ), mỗi cái trung bình trên q, k, v."""
        cosine, kl = student_qkv.new_zeros(()), student_qkv.new_zeros(())
        for s, t in zip(student_qkv, teacher_qkv):
            cosine = cosine + (1.0 - F.cosine_similarity(s, t, dim=-1)).mean()
            if self.rel_weight > 0:
                kl = kl + F.kl_div(F.log_softmax(relation(s, self.heads), dim=-1),
                                   F.softmax(relation(t, self.heads), dim=-1), reduction="batchmean")
        return cosine / 3, kl / 3


class CRDLoss(nn.Module):
    """Contrastive Representation Distillation với memory bank cho hai view, đối xứng teacher <-> student."""

    def __init__(self, student_dim: int, teacher_dim: int, num_samples: int, embed_dim: int, negatives: int,
                 temperature: float, momentum: float) -> None:
        super().__init__()
        self.embed_student = nn.Linear(student_dim, embed_dim)
        self.embed_teacher = nn.Linear(teacher_dim, embed_dim)
        self.negatives, self.temperature, self.momentum = negatives, temperature, momentum
        self.register_buffer("bank_student", F.normalize(torch.randn(num_samples, embed_dim), dim=1))
        self.register_buffer("bank_teacher", F.normalize(torch.randn(num_samples, embed_dim), dim=1))
        self.register_buffer("bank_labels", torch.full((num_samples,), -1, dtype=torch.long))

    def _nce(self, anchor: Tensor, positive: Tensor, bank: Tensor, negative_index: Tensor, index: Tensor, labels: Tensor) -> Tensor:
        positive_logit = (anchor * positive).sum(dim=1, keepdim=True)
        negative_logit = torch.einsum("be,bke->bk", anchor, bank[negative_index])
        invalid = (self.bank_labels[negative_index] == labels[:, None]) | (negative_index == index[:, None])
        negative_logit = negative_logit.masked_fill(invalid, float("-inf"))
        logits = torch.cat([positive_logit, negative_logit], dim=1) / self.temperature
        return F.cross_entropy(logits, torch.zeros(len(anchor), dtype=torch.long, device=anchor.device))

    def forward(self, student_embedding: Tensor, teacher_embedding: Tensor, index: Tensor, labels: Tensor) -> Tensor:
        zs = F.normalize(self.embed_student(student_embedding), dim=1)
        zt = F.normalize(self.embed_teacher(teacher_embedding), dim=1)
        negative_index = torch.randint(0, self.bank_student.shape[0], (len(index), self.negatives), device=index.device)
        loss = (self._nce(zs, zt, self.bank_teacher, negative_index, index, labels)
                + self._nce(zt, zs, self.bank_student, negative_index, index, labels))
        with torch.no_grad():
            for bank, z in ((self.bank_student, zs), (self.bank_teacher, zt)):
                bank[index] = F.normalize(self.momentum * bank[index] + (1.0 - self.momentum) * z.detach(), dim=1)
            self.bank_labels[index] = labels
        return loss

In [ ]:
# (5b) Tải dataset mixture-36l từ Kaggle (nếu chưa có) và kiểm tra đủ tệp. Token đọc từ KAGGLE_ACCESS_TOKEN, không ghi vào notebook.
import csv
import os
import time

from scripts import download_dataset as dl
from scripts.load_dataset import MIXTURE_KEY

DOWNLOAD_ROOT = ROOT / "datasets"


def ensure_mixture36(download_root: Path) -> Path:
    print(f"[1/3] tìm {MIXTURE_KEY} (env, url.yaml), nếu thiếu sẽ tải từ Kaggle vào {download_root}")
    if not os.environ.get(dl.TOKEN_ENV):
        print(f"      chưa đặt {dl.TOKEN_ENV}: chỉ dùng được dataset đã có trên máy")
    started = time.perf_counter()
    root = dl.ensure_dataset(MIXTURE_KEY, download_root)  # thanh tqdm: byte khi tải, tệp khi giải nén
    print(f"[2/3] dataset tại {root} ({time.perf_counter() - started:.0f}s)")
    print("[3/3] kiểm tra tệp mixture và noise theo manifest")
    for split in ("train", "validation", "test"):
        with open(root / split / "manifest.csv", encoding="utf-8-sig", newline="") as f:
            rows = list(csv.DictReader(f))
        missing = sum(not (root / row[key]).is_file() for row in tqdm(rows, desc=split, unit="mẫu") for key in ("mixture_path", "noise_path"))
        print(f"      {split}: {len(rows)} mẫu, thiếu {missing} tệp")
        if missing:
            raise FileNotFoundError(f"{split}: thiếu {missing} tệp, xoá thư mục {root} rồi tải lại")
    return root


DATASET_ROOT = ensure_mixture36(DOWNLOAD_ROOT)

In [ ]:
# (6) Dữ liệu: train có nhiễu thật và chỉ số mẫu (cho memory bank CRD)
from scripts.finetune import SMOKE_BATCH, SMOKE_SAMPLES
from scripts.load_dataset import Mixture36Dataset, build_dataloader, subsample

DURATION = 4.0


def nhat_path(split: str) -> Path:
    return NHAT_CACHE_DIR / f"{split}.npy"


class NoiseCache:
    """n^ đã tính trước, đọc qua memmap; mỗi worker mở riêng."""

    def __init__(self, split: str) -> None:
        self.path, self._array = nhat_path(split), None

    def __getitem__(self, index: int) -> Tensor:
        if self._array is None:
            self._array = np.load(self.path, mmap_mode="r")
        return torch.from_numpy(np.array(self._array[index]))


class IndexedDataset(Dataset):
    def __init__(self, base: Dataset, cache: NoiseCache | None = None) -> None:
        self.base, self.cache = base, cache

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, index: int) -> dict:
        item = dict(self.base[index])
        item["index"] = torch.tensor(index)
        if self.cache is not None:
            original = self.base.indices[index] if isinstance(self.base, Subset) else index
            item["noise_hat"] = self.cache[original]
        return item


def with_fork(loader):
    """Kernel marimo mặc định tạo worker bằng spawn, không pickle được IndexedDataset (định nghĩa trong cell). Worker chỉ đọc tệp, không dùng CUDA, nên fork an toàn."""
    if loader.num_workers > 0:
        loader.multiprocessing_context = "fork"
    return loader


def rms(x: Tensor) -> Tensor:
    return x.pow(2).mean(dim=-1, keepdim=True).sqrt().clamp_min(1e-8)


def augment_pair(noise_hat: Tensor, noise: Tensor, labels: Tensor, aug: dict) -> tuple[Tensor, Tensor, Tensor, bool]:
    """Biến đổi NHẤT QUÁN cặp (n^ cho student, nhiễu thật cho teacher) để teacher vẫn cho nhãn và q/k/v đúng cho đầu vào của student.

    1. Đưa từng tín hiệu về RMS = 1. Năng lượng n^ lệch so với nhiễu thật tuỳ SNR và tuỳ clip (FRCRN giữ khoảng 95% ở -5 dB nhưng chỉ
       khoảng 65% ở 20 dB), nên nếu trộn thẳng thì cân bằng giữa hai nguồn ở student và ở teacher sẽ khác nhau. Gain đơn thuần thì không
       ảnh hưởng vì log-mel và teacher đều chuẩn hoá theo clip, nên không cần gain ngẫu nhiên riêng.
    2. Dịch vòng cùng một lượng cho cả hai tín hiệu.
    3. Mixup: trộn với cùng hệ số lam (lam >= 0.5, mẫu gốc luôn chiếm ưu thế) và cùng hoán vị; nhãn mềm lam*y_i + (1-lam)*y_j.
    """
    noise_hat, noise = noise_hat / rms(noise_hat), noise / rms(noise)
    batch, samples = noise.shape
    if aug["shift"]:
        start = torch.randint(0, samples, (batch, 1), device=noise.device)
        index = (torch.arange(samples, device=noise.device)[None, :] + start) % samples
        noise_hat, noise = noise_hat.gather(1, index), noise.gather(1, index)
    targets = F.one_hot(labels, NUM_CLASSES).float()
    mixed = bool(aug["mixup_prob"] > 0 and torch.rand(()) < aug["mixup_prob"])
    if mixed:
        lam = torch.distributions.Beta(aug["mixup_alpha"], aug["mixup_alpha"]).sample((batch, 1)).to(noise.device)
        lam = torch.maximum(lam, 1.0 - lam)
        order = torch.randperm(batch, device=noise.device)
        noise_hat = lam * noise_hat + (1.0 - lam) * noise_hat[order]
        noise = lam * noise + (1.0 - lam) * noise[order]
        targets = lam * targets + (1.0 - lam) * targets[order]
    return noise_hat, noise, targets, mixed


def use_cache() -> bool:
    return USE_NHAT_CACHE and RUN_TRAINING  # smoke test không dùng cache


def cache_ready(split: str) -> bool:
    return use_cache() and (NHAT_CACHE_DIR / f"{split}.done").is_file()  # test không cache: chỉ dùng một lần, FRCRN chạy trực tiếp


def build_eval_loader(split: str, batch_size: int, workers: int, max_samples: int | None = None):
    base = subsample(Mixture36Dataset(split=split, duration=DURATION), max_samples)
    dataset = IndexedDataset(base, NoiseCache(split)) if cache_ready(split) else base
    return with_fork(build_dataloader(dataset, batch_size, False, workers))


def build_loaders(train_config: dict, max_samples: int | None, workers: int) -> dict:
    base = subsample(Mixture36Dataset(split="train", duration=DURATION, load_noise=True), max_samples)
    train_ds = IndexedDataset(base, NoiseCache("train") if use_cache() else None)
    return {
        "train": with_fork(build_dataloader(train_ds, train_config["batch_size"], True, workers)),
        "validation": build_eval_loader("validation", train_config["eval_batch_size"], workers, max_samples),
        "num_train": len(train_ds),
    }

In [ ]:
# (6b) Tính trước n^ = FRCRN(mixture) cho train và validation (test chỉ dùng một lần nên chạy FRCRN trực tiếp). Chạy một lần (khoảng 17 phút), lần sau bỏ qua nhờ tệp .done.
import zipfile

from numpy.lib.format import open_memmap


def build_nhat_cache(splits=("train", "validation")) -> None:
    NHAT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    separator = student.separator.eval()
    for split in splits:
        done, lock = NHAT_CACHE_DIR / f"{split}.done", NHAT_CACHE_DIR / f"{split}.building"
        dataset = Mixture36Dataset(split=split, duration=DURATION)
        if done.is_file() and done.read_text() == str(len(dataset)):
            print(f"n^ cache {split}: đã có ({len(dataset)} clip), bỏ qua")
            continue
        if lock.is_file() and time.time() - lock.stat().st_mtime < 600:  # hai tiến trình cùng ghi một tệp đã làm hỏng cache một lần
            raise RuntimeError(f"đang có tiến trình khác tính n^ {split} ({lock}), đợi nó xong rồi chạy lại")
        lock.write_text("building")
        samples = int(DURATION * FRONTEND["sample_rate"])
        out = open_memmap(nhat_path(split), mode="w+", dtype=np.float32, shape=(len(dataset), samples))
        position, started = 0, time.perf_counter()
        for step, batch in enumerate(tqdm(with_fork(build_dataloader(dataset, 64, False, NUM_WORKERS)), desc=f"n^ {split}", unit="batch")):
            with torch.no_grad():
                estimate = separator(batch["waveform"].to(device).float()).cpu().numpy()
            out[position:position + len(estimate)] = estimate
            position += len(estimate)
            if step % 20 == 0:
                lock.touch()
        out.flush()
        done.write_text(str(len(dataset)))
        lock.unlink(missing_ok=True)
        print(f"n^ cache {split}: {len(dataset)} clip trong {time.perf_counter() - started:.0f}s -> {nhat_path(split)}")


def empty_rows(split: str) -> np.ndarray:
    """Hàng toàn số 0 trong cache (chưa được ghi)."""
    array, rows = np.load(nhat_path(split), mmap_mode="r"), []
    for start in range(0, len(array), 512):
        energy = np.square(np.asarray(array[start:start + 512], dtype=np.float64)).sum(axis=1)
        rows.extend((start + np.nonzero(energy == 0)[0]).tolist())
    return np.array(rows, dtype=np.int64)


def fix_empty_rows(split: str) -> None:
    rows = empty_rows(split)
    if not len(rows):
        print(f"n^ cache {split}: không có hàng trống")
        return
    print(f"n^ cache {split}: {len(rows)} hàng trống ({len(rows) / len(np.load(nhat_path(split), mmap_mode='r')):.1%}), tính lại riêng các hàng này")
    array = np.load(nhat_path(split), mmap_mode="r+")
    subset = Subset(Mixture36Dataset(split=split, duration=DURATION), rows.tolist())
    position = 0
    for batch in tqdm(with_fork(build_dataloader(subset, 64, False, NUM_WORKERS)), desc=f"n^ sửa {split}", unit="batch"):
        with torch.no_grad():
            estimate = student.separator.eval()(batch["waveform"].to(device).float()).cpu().numpy()
        array[rows[position:position + len(estimate)]] = estimate
        position += len(estimate)
    array.flush()
    assert not len(empty_rows(split)), f"n^ cache {split}: vẫn còn hàng trống sau khi sửa"


def check_nhat_cache(count: int = 8) -> None:
    probe = Mixture36Dataset(split="train", duration=DURATION)
    waveform = torch.stack([probe[i]["waveform"] for i in range(count)]).to(device)
    with torch.no_grad():
        live = student.separator(waveform.float()).cpu()
    cached = torch.stack([NoiseCache("train")[i] for i in range(count)])
    error = float((live - cached).abs().max())
    snr = float(10 * torch.log10(live.pow(2).sum() / (live - cached).pow(2).sum().clamp_min(1e-20)))
    print(f"n^ cache: so với FRCRN chạy trực tiếp, sai lệch lớn nhất {error:.2e} (biên độ tối đa {float(live.abs().max()):.3f}), SNR giữa hai bản {snr:.1f} dB")
    # cuDNN TF32 không tất định và phụ thuộc cỡ batch (cùng batch: khoảng 90 dB, khác batch: khoảng 60 dB), nên so bằng SNR chứ không bằng sai số tuyệt đối
    assert snr > 40, "cache n^ không khớp FRCRN trực tiếp"


def nhat_ready() -> bool:
    return all((NHAT_CACHE_DIR / f"{name}.done").is_file() for name in ("train", "validation"))


def restore_nhat_from_drive() -> None:
    """Nếu thiếu cache trên máy: tải zip từ Drive rồi giải nén, thay cho việc tính lại bằng FRCRN."""
    if nhat_ready() or fs is None:
        return
    try:
        if not fs.exists(NHAT_DRIVE_ZIP):
            print("n^ cache: Drive chưa có bản lưu, sẽ tính bằng FRCRN")
            return
        print(f"n^ cache: tải {NHAT_DRIVE_ZIP} từ Drive (khoảng 8,6 GB)")
        started = time.perf_counter()
        fs.get_file(NHAT_DRIVE_ZIP, str(NHAT_ZIP))
        with zipfile.ZipFile(NHAT_ZIP) as archive:
            archive.extractall(ROOT)  # các mục có dạng cache/nhat/*.npy
        print(f"n^ cache: đã tải và giải nén trong {time.perf_counter() - started:.0f}s")
    except Exception as error:
        print("drive: không khôi phục được cache n^, sẽ tính bằng FRCRN:", repr(error))


def save_nhat_to_drive() -> None:
    """Đẩy zip cache lên Drive một lần (bỏ qua nếu Drive đã có). Thư mục tạo một lần, tránh makedirs lặp lại sinh thư mục trùng tên."""
    if fs is None or not UPLOAD_NHAT_TO_DRIVE or not nhat_ready():
        return
    try:
        if fs.exists(NHAT_DRIVE_ZIP):
            print("n^ cache: Drive đã có bản lưu, bỏ qua upload")
            return
        if not NHAT_ZIP.is_file():
            print("n^ cache: nén thành", NHAT_ZIP)
            partial = NHAT_ZIP.with_name(NHAT_ZIP.name + ".part")
            with zipfile.ZipFile(partial, "w", zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
                for path in sorted(NHAT_CACHE_DIR.glob("*")):
                    archive.write(path, path.relative_to(ROOT).as_posix())
            partial.replace(NHAT_ZIP)
        fs.makedirs(NHAT_DRIVE_ZIP.rsplit("/", 1)[0], exist_ok=True)
        print(f"n^ cache: upload {NHAT_ZIP.stat().st_size / 1e9:.1f} GB lên Drive ({NHAT_DRIVE_ZIP})")
        started = time.perf_counter()
        fs.put_file(str(NHAT_ZIP), NHAT_DRIVE_ZIP)
        print(f"n^ cache: upload xong sau {time.perf_counter() - started:.0f}s")
    except Exception as error:
        print("drive: upload cache n^ lỗi (huấn luyện vẫn tiếp tục):", repr(error))


if use_cache():
    restore_nhat_from_drive()
    build_nhat_cache()
    for split_name in ("train", "validation"):
        fix_empty_rows(split_name)  # chạy lại lần sau vẫn an toàn: không có hàng trống thì bỏ qua (quét khoảng 10 giây)
    check_nhat_cache()
    save_nhat_to_drive()
else:
    print("n^ cache: tắt (USE_NHAT_CACHE=False hoặc smoke test), FRCRN chạy trực tiếp mỗi bước")

## Kiểm tra teacher (tuỳ chọn, rẻ)

Teacher phải phân loại nhiễu thật đúng cỡ `clean_val_accuracy` trong checkpoint (fold 0: 0.74). Nếu thấp hơn nhiều thì đường vào (RMS, fbank) lệch so với lúc fine-tune, nên dừng lại trước khi huấn luyện student.

In [ ]:
# (7) Độ chính xác của teacher trên nhiễu thật của tập validation
@torch.no_grad()
def check_teacher(max_batches: int = 20, batch_size: int = 32) -> float:
    loader = build_dataloader(Mixture36Dataset(split="validation", duration=DURATION, load_noise=True), batch_size, False, 0)
    correct = seen = 0
    for step, batch in enumerate(loader):
        if step >= max_batches:
            break
        output = teacher(batch["noise"].to(device))
        correct += (output.logits.argmax(1).cpu() == batch["target"].argmax(1)).sum().item()
        seen += len(batch["target"])
    logger.info("teacher: accuracy trên nhiễu thật (validation, %d mẫu) = %.4f", seen, correct / seen)
    return correct / seen


# check_teacher()   # bỏ comment để chạy

## Huấn luyện và đánh giá

In [ ]:
# (8) Vòng huấn luyện + đánh giá 9 chỉ số theo từng SNR
from src.metrics import ClassificationMetrics  # noqa: F401  (snr_scores dùng bên dưới)
from torch.optim.lr_scheduler import ReduceLROnPlateau
from src.train.reviewkd_trainer import NINE_METRICS, snr_scores


class DrivePut:
    """Upload một tệp lên Drive, ghi đè nếu đã có. Thư mục chỉ tạo MỘT lần (ensure): makedirs lặp lại ở mỗi lần upload đã sinh thư mục trùng tên
    trên Drive và gây MultipleFilesError. Lỗi mạng chỉ ghi cảnh báo, không làm gãy huấn luyện."""

    ready: set[str] = set()

    def __init__(self, fs, remote_dir: str, filename: str, logger, attempts: int = 3) -> None:
        self.fs, self.remote_dir, self.filename, self.logger, self.attempts = fs, remote_dir.strip("/"), filename, logger, attempts

    @classmethod
    def ensure(cls, fs, remote_dir: str, logger) -> bool:
        remote_dir = remote_dir.strip("/")
        if remote_dir in cls.ready:
            return True
        try:
            fs.makedirs(remote_dir, exist_ok=True)
            cls.ready.add(remote_dir)
            return True
        except Exception as error:
            logger.warning("drive: không tạo được thư mục %s: %r", remote_dir, error)
            return False

    def __call__(self, local_path, epoch: int | None = None, score: float | None = None) -> None:
        if self.remote_dir not in self.ready:
            return  # thư mục chưa sẵn sàng thì bỏ qua, không thử tạo lại
        target = f"{self.remote_dir}/{self.filename}"
        for attempt in range(1, self.attempts + 1):
            try:
                self.fs.put_file(str(local_path), target)
                self.logger.info("drive backup: %s -> %s", Path(local_path).name, target)
                return
            except Exception as error:
                if attempt == self.attempts:
                    self.logger.warning("drive backup failed for %s: %r", local_path, error)
                else:
                    time.sleep(5 * attempt)
from src.metrics import _binary_ap, _binary_auc, confusion_matrix, top_confusions  # hàm nội bộ của repo


def trainable_state(model: nn.Module) -> dict[str, Tensor]:
    """Trọng số cần lưu: backbone (net) và attention pooling (att, cla). Bỏ separator (FRCRN đóng băng, nạp từ pretrained).
    Trước đây chỉ lưu model.net nên thiếu att/cla: best.pt không phân loại được và resume làm mất lớp phân loại."""
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items() if not key.startswith("separator.")}


def load_trainable_state(model: nn.Module, state: dict[str, Tensor]) -> None:
    result = model.load_state_dict(state, strict=False)
    problems = [key for key in result.missing_keys if not key.startswith("separator.")] + list(result.unexpected_keys)
    if problems:
        raise ValueError(f"trọng số không khớp mô hình (thiếu hoặc thừa): {problems[:6]}")


def save_model(path: Path, model: nn.Module, **metadata) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": trainable_state(model), **metadata}, path)
    return path


@torch.no_grad()
def evaluate(model: FRCRNMobileViT, loader, desc: str = "evaluate") -> dict[str, float]:
    model.eval()
    probabilities, labels, snrs = [], [], []
    for batch in tqdm(loader, desc=desc, unit="batch", leave=False, dynamic_ncols=True):
        if "noise_hat" in batch:
            probabilities.append(model.predict_proba(noise_hat=batch["noise_hat"].to(device)).float().cpu())
        else:
            probabilities.append(model.predict_proba(batch["waveform"].to(device)).float().cpu())
        labels.append(batch["target"].argmax(dim=1))
        snrs.append(batch["snr_db"])
    return snr_scores(torch.cat(probabilities).numpy(), torch.cat(labels).numpy(), torch.cat(snrs).numpy(), NUM_CLASSES)


def per_snr_table(scores: dict[str, float]) -> pd.DataFrame:
    rows = {f"{level}": [scores.get(f"{name}_snr_{level}", float("nan")) for name in NINE_METRICS] for level in SNR_LEVELS}
    rows["mean"] = [scores.get(f"{name}_snr_mean", float("nan")) for name in NINE_METRICS]
    return pd.DataFrame.from_dict(rows, orient="index", columns=NINE_METRICS).rename_axis("snr_db")


@torch.no_grad()
def analyze_predictions(model: nn.Module, loader, split: str, name: str, out_dir: Path, class_names: list[str]) -> list[Path]:
    """Dự đoán của mô hình tốt nhất: xác suất từng clip, ma trận nhầm lẫn, bảng theo từng lớp (recall, precision, AP, AUC, lớp hay bị nhầm thành, recall theo SNR)."""
    model.eval()
    probs, labels, snrs = [], [], []
    for batch in tqdm(loader, desc=f"{name} - dự đoán {split}", unit="batch", leave=False, dynamic_ncols=True):
        if "noise_hat" in batch:
            probs.append(model.predict_proba(noise_hat=batch["noise_hat"].to(device)).float().cpu())
        else:
            probs.append(model.predict_proba(batch["waveform"].to(device)).float().cpu())
        labels.append(batch["target"].argmax(dim=1))
        snrs.append(batch["snr_db"])
    P, y, snr = torch.cat(probs).numpy(), torch.cat(labels).numpy(), torch.cat(snrs).numpy()
    pred = P.argmax(axis=1)
    matrix = confusion_matrix(y, pred, NUM_CLASSES)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = [out_dir / f"predictions_{split}_{name}.npz", out_dir / f"confusion_{split}_{name}.csv", out_dir / f"per_class_{split}_{name}.csv"]
    np.savez_compressed(paths[0], probs=P.astype(np.float16), labels=y, snr_db=snr, class_names=np.array(class_names))
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(paths[1])
    rows = []
    for c in range(NUM_CLASSES):
        off_diagonal = matrix[c].copy()
        off_diagonal[c] = 0
        partner, total = int(off_diagonal.argmax()), max(int(matrix[c].sum()), 1)
        row = dict(label=class_names[c], clips=int(matrix[c].sum()), recall=matrix[c, c] / total, precision=matrix[c, c] / max(int(matrix[:, c].sum()), 1),
                   AP=_binary_ap((y == c).astype(np.int64), P[:, c]), AUC=_binary_auc((y == c).astype(np.int64), P[:, c]),
                   most_confused_as=class_names[partner], confused_rate=off_diagonal[partner] / total)
        for level in SNR_LEVELS:
            selected = (y == c) & (snr == level)
            row[f"recall_snr_{level}"] = float((pred[selected] == c).mean()) if selected.any() else float("nan")
        rows.append(row)
    table = pd.DataFrame(rows).sort_values("recall")
    table.to_csv(paths[2], index=False)
    logger.info("%s %s: 12 cặp nhầm nhiều nhất (thật -> đoán, tỉ lệ trên lớp thật)\n%s", name, split,
                "\n".join(f"  {p.true_label} -> {p.predicted_label}: {p.rate:.0%} ({p.count} clip)" for p in top_confusions(matrix, class_names, 12)))
    logger.info("%s %s: 8 lớp yếu nhất theo recall\n%s", name, split, table.head(8)[["label", "recall", "precision", "AP", "most_confused_as", "confused_rate"]].round(3).to_string(index=False))
    return paths


def kd_scale(epoch: int, warmup: int) -> float:
    return 1.0 if warmup == 0 else min(1.0, (epoch - 1) / warmup)


def run_experiment(name: str) -> tuple[dict, pd.DataFrame | None]:
    train_config, kd = dict(TRAIN), dict(KD)
    max_samples, workers = None, NUM_WORKERS
    if not RUN_TRAINING:
        train_config.update(epochs=1, patience=1, batch_size=SMOKE_BATCH, eval_batch_size=SMOKE_BATCH)
        kd["warmup_epochs"] = 0
        max_samples, workers = SMOKE_SAMPLES, 0

    set_seed(train_config["seed"])
    loaders = build_loaders(train_config, max_samples, workers)
    model = build_student().to(device)
    qkv_loss = QKVLoss(heads=12, rel_weight=kd["rel_weight"]).to(device)
    crd = (CRDLoss(EMBED_CHANNELS, 768, loaders["num_train"], kd["crd_dim"], kd["crd_negatives"],
                   kd["crd_temperature"], kd["crd_momentum"]).to(device) if USE_CRD else None)

    trainable = [p for module in (model, crd) if module is not None for p in module.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=train_config["lr"], weight_decay=train_config["weight_decay"], betas=train_config["betas"])
    steps_per_epoch = len(loaders["train"])
    warmup_steps = max(int(train_config["epochs"] * steps_per_epoch * train_config["warmup_ratio"]), 1)  # warm-up tuyến tính theo bước
    plateau = ReduceLROnPlateau(optimizer, mode="max", factor=train_config["lr_factor"], patience=train_config["lr_patience"],
                                threshold=train_config["lr_threshold"], threshold_mode="abs", min_lr=train_config["min_lr"])
    for group in optimizer.param_groups:
        group["lr"] = train_config["lr"] / warmup_steps  # bước đầu tiên của warm-up; nạp resume sẽ ghi đè bằng lr đã lưu
    logger.info("%s | student %.2fM | crd %.2fM | train %d mẫu", name,
                sum(p.numel() for p in model.net.parameters()) / 1e6,
                sum(p.numel() for p in crd.parameters()) / 1e6 if crd else 0.0, loaders["num_train"])

    history, best_score, best_epoch, best_state, stale, start_epoch = [], -float("inf"), 0, None, 0, 1
    flags = dict(qkv=USE_QKV, crd=USE_CRD, logit=USE_LOGIT, layers=TEACHER_LAYERS, student=STUDENT, kd=kd,
                 reg=dict(dropout=TRAIN["dropout"], drop_path=TRAIN["drop_path"], wd=TRAIN["weight_decay"]),
                 aug=(AUG if USE_AUG else None),
                 opt=dict(lr=TRAIN["lr"], factor=TRAIN["lr_factor"], patience=TRAIN["lr_patience"], threshold=TRAIN["lr_threshold"],
                          min_lr=TRAIN["min_lr"], warmup=TRAIN["warmup_ratio"]), cache=use_cache(), fmt=2)
    resume_path = ROOT / "output" / "resume" / name / "resume.pt" if RUN_TRAINING else None  # thư mục cố định để Run all nối tiếp được
    remote_dir = f"{DRIVE_DIR}/{name}"
    use_drive = fs is not None and RUN_TRAINING
    drive_ready = use_drive and DrivePut.ensure(fs, remote_dir, logger)  # tạo thư mục Drive một lần duy nhất
    drive_best = DrivePut(fs, remote_dir, "best.pt", logger) if drive_ready else None
    drive_resume = DrivePut(fs, remote_dir, "resume.pt", logger) if drive_ready else None
    drive_history = DrivePut(fs, remote_dir, f"history_{name}.csv", logger) if drive_ready else None
    drive_logfile = DrivePut(fs, remote_dir, run.log_file.name, logger) if drive_ready else None
    if use_drive and not resume_path.is_file():  # local mất (sandbox mới) thì lấy resume.pt từ Drive
        try:
            if fs.exists(f"{remote_dir}/resume.pt"):
                resume_path.parent.mkdir(parents=True, exist_ok=True)
                fs.get_file(f"{remote_dir}/resume.pt", str(resume_path))
                logger.info("drive: khôi phục resume.pt từ %s", remote_dir)
        except Exception as error:
            logger.warning("drive: không khôi phục được resume.pt: %r", error)
    if resume_path is not None and resume_path.is_file():
        state = torch.load(resume_path, map_location="cpu", weights_only=False)
        if state["flags"] != flags:
            raise ValueError(f"{resume_path}: cấu hình khác lần chạy trước ({state['flags']} != {flags}), xoá file để chạy lại từ đầu")
        load_trainable_state(model, state["model"])
        if crd is not None:
            crd.load_state_dict(state["crd"])  # gồm cả memory bank
        optimizer.load_state_dict(state["optimizer"])
        plateau.load_state_dict(state["scheduler"])
        history, best_score, best_epoch, best_state, stale = state["history"], state["best_score"], state["best_epoch"], state["best_state"], state["stale"]
        start_epoch = state["epoch"] + 1
        set_seed(train_config["seed"] + start_epoch)
        logger.info("%s: nối tiếp từ epoch %d (best %.4f ở epoch %d, stale %d)", name, start_epoch, best_score, best_epoch, stale)
    for epoch in range(start_epoch, train_config["epochs"] + 1):
        model.train()
        scale = kd_scale(epoch, kd["warmup_epochs"])
        sums = dict(loss=0.0, ce=0.0, cos=0.0, rel=0.0, crd=0.0, kl=0.0, correct=0, seen=0)
        cos_layer_sum = None  # 1 - cos(q,k,v) của từng lớp, để thấy lớp nào học chậm
        for batch_index, batch in enumerate(tqdm(loaders["train"], desc=f"epoch {epoch}", unit="batch", leave=False, dynamic_ncols=True)):
            step_index = (epoch - 1) * steps_per_epoch + batch_index
            if step_index < warmup_steps:
                for group in optimizer.param_groups:
                    group["lr"] = train_config["lr"] * (step_index + 1) / warmup_steps
            noise = batch["noise"].to(device)
            labels, index = batch["target"].argmax(dim=1).to(device), batch["index"].to(device)
            noise_hat, targets, mixed = (batch["noise_hat"].to(device) if "noise_hat" in batch else None), labels, False
            if USE_AUG and noise_hat is not None:
                noise_hat, noise, targets, mixed = augment_pair(noise_hat, noise, labels, AUG)
            output = model(noise_hat=noise_hat) if noise_hat is not None else model(batch["waveform"].to(device))
            ce = F.cross_entropy(output.logits, targets, label_smoothing=train_config["label_smoothing"])
            loss, cos, rel, nce, kl = ce, ce.new_zeros(()), ce.new_zeros(()), ce.new_zeros(()), ce.new_zeros(())
            if USE_QKV or USE_CRD or USE_LOGIT:
                teacher_output = teacher(noise)
            if USE_QKV:
                pairs = [qkv_loss(output.qkv[i], pool_teacher(teacher_output.qkv[i], GRID, output.grids[i])) for i in range(len(output.qkv))]
                cos_layers, rel_layers = torch.stack([p[0] for p in pairs]), torch.stack([p[1] for p in pairs])
                cos, rel = cos_layers.mean(), rel_layers.mean()
                step_layers = cos_layers.detach() * len(labels)
                cos_layer_sum = step_layers if cos_layer_sum is None else cos_layer_sum + step_layers
                loss = loss + scale * kd["qkv_weight"] * (cos + kd["rel_weight"] * rel)
            if USE_CRD and not mixed:  # batch trộn không có chỉ số mẫu và nhãn đơn nên không cập nhật memory bank của CRD
                nce = crd(output.embedding, teacher_output.embedding, index, labels)
                loss = loss + scale * kd["crd_weight"] * nce
            if USE_LOGIT:
                temperature = kd["logit_temperature"]
                kl = F.kl_div(F.log_softmax(output.logits / temperature, dim=-1),
                              F.softmax(teacher_output.logits / temperature, dim=-1), reduction="batchmean") * temperature**2
                loss = loss + scale * kd["logit_weight"] * kl
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(trainable, train_config["grad_clip"])
            optimizer.step()
            n = len(labels)
            for key, value in (("loss", loss), ("ce", ce), ("cos", cos), ("rel", rel), ("crd", nce), ("kl", kl)):
                sums[key] += value.item() * n
            sums["correct"] += (output.logits.argmax(1) == labels).sum().item()
            sums["seen"] += n
        seen = max(sums["seen"], 1)
        scores = evaluate(model, loaders["validation"], desc=f"epoch {epoch} - validation")
        record = {"epoch": epoch, "kd_scale": scale, "lr": optimizer.param_groups[0]["lr"], "train_acc": sums["correct"] / seen,
                  **{f"train_{k}": sums[k] / seen for k in ("loss", "ce", "cos", "rel", "crd", "kl")},
                  **({f"train_cos_l{i + 1}": v / seen for i, v in enumerate(cos_layer_sum.tolist())} if cos_layer_sum is not None else {}),
                  **{f"val_{k}": v for k, v in scores.items()}}
        history.append(record)
        logger.info("%s epoch %d | loss %.4f ce %.4f cos %.4f rel %.4f crd %.4f kl %.4f | train acc %.4f | val macro_acc %.4f mAP_macro %.4f | lr %.2e",
                    name, epoch, record["train_loss"], record["train_ce"], record["train_cos"], record["train_rel"],
                    record["train_crd"], record["train_kl"], record["train_acc"], scores["macro_acc"], scores["mAP_macro"], record["lr"])
        if (epoch * steps_per_epoch) >= warmup_steps:  # chỉ giảm lr sau warm-up
            plateau.step(scores["macro_acc"])
            if optimizer.param_groups[0]["lr"] < record["lr"]:
                logger.info("%s: val macro_acc không cải thiện, giảm lr %.2e -> %.2e", name, record["lr"], optimizer.param_groups[0]["lr"])
        if RUN_TRAINING:
            run.result_dir.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(history).to_csv(run.result_dir / f"history_{name}.csv", index=False)
            if drive_history is not None:
                drive_history(run.result_dir / f"history_{name}.csv")
        if scores["macro_acc"] > best_score:
            best_score, best_epoch, stale = scores["macro_acc"], epoch, 0
            best_state = trainable_state(model)
            if RUN_TRAINING:  # lưu ngay khi tốt hơn, để sandbox mất kết nối giữa chừng vẫn còn checkpoint
                save_model(run.checkpoint_path(name), model, epoch=epoch, macro_acc=best_score, frcrn=str(FRCRN_PATH.relative_to(ROOT)))
                if drive_best is not None:
                    drive_best(run.checkpoint_path(name), epoch, best_score)  # ghi đè best.pt trên Drive
        else:
            stale += 1
        if resume_path is not None:  # ghi nguyên tử, để bị ngắt giữa lúc ghi vẫn còn bản cũ
            resume_path.parent.mkdir(parents=True, exist_ok=True)
            partial = resume_path.with_name(resume_path.name + ".part")
            torch.save({"epoch": epoch, "flags": flags, "model": trainable_state(model),
                        "crd": crd.state_dict() if crd is not None else None,
                        "optimizer": optimizer.state_dict(), "scheduler": plateau.state_dict(), "history": history,
                        "best_score": best_score, "best_epoch": best_epoch, "best_state": best_state, "stale": stale}, partial)
            partial.replace(resume_path)
            if drive_resume is not None:
                drive_resume(resume_path)
            if drive_logfile is not None:
                drive_logfile(run.log_file)
        if stale >= train_config["patience"]:
            logger.info("%s: dừng sớm ở epoch %d (best %d, %.4f)", name, epoch, best_epoch, best_score)
            break

    load_trainable_state(model, best_state)
    val_table = per_snr_table({k.removeprefix("val_"): v for k, v in history[best_epoch - 1].items() if k.startswith("val_")})
    logger.info("%s: validation tại epoch tốt nhất %d, 9 chỉ số theo SNR\n%s", name, best_epoch, val_table.round(4).to_string())
    class_names = Mixture36Dataset(split="validation", duration=DURATION).class_names
    analysis_files = analyze_predictions(model, loaders["validation"], "validation", name, run.result_dir, class_names)
    test_table = None
    if RUN_TRAINING:
        test_loader = build_eval_loader("test", train_config["eval_batch_size"], workers)
        test_table = per_snr_table(evaluate(model, test_loader, desc=f"{name} - test"))
        analysis_files += analyze_predictions(model, test_loader, "test", name, run.result_dir, class_names)
        logger.info("%s: TEST, 9 chỉ số theo SNR\n%s", name, test_table.round(4).to_string())
        checkpoint_path = run.checkpoint_path(name)
        save_model(checkpoint_path, model, epoch=best_epoch, macro_acc=best_score, frcrn=str(FRCRN_PATH.relative_to(ROOT)))
        run.result_dir.mkdir(parents=True, exist_ok=True)
        val_table.to_csv(run.result_dir / f"per_snr_validation_{name}.csv")
        test_table.to_csv(run.result_dir / f"per_snr_test_{name}.csv")
        if use_drive:
            for table_file in (f"per_snr_validation_{name}.csv", f"per_snr_test_{name}.csv"):
                DrivePut(fs, remote_dir, table_file, logger)(run.result_dir / table_file)
            for analysis_file in analysis_files:
                DrivePut(fs, remote_dir, analysis_file.name, logger)(analysis_file)
        pd.DataFrame(history).to_csv(run.result_dir / f"history_{name}.csv", index=False)
    if drive_logfile is not None:
        drive_logfile(run.log_file)
    return {"model": name, "best_epoch": best_epoch, "val_macro_acc": best_score}, (test_table if test_table is not None else val_table)

In [ ]:
# (9) Chạy một thí nghiệm theo các cờ ở cell (1). RUN_TRAINING = True huấn luyện đầy đủ, False chạy smoke test nhỏ.
# Chỉ chạy cell này khi đã đồng ý dùng máy.
name = "-".join(["mobilevit12", f"ffn{STUDENT['ffn_dim']}", f"lr{TRAIN['lr']:.0e}-do{round(TRAIN['dropout'] * 100)}-dp{round(TRAIN['drop_path'] * 100)}-wd{round(TRAIN['weight_decay'] * 100)}", "frcrn", *([f"mx{round(AUG['mixup_prob'] * 100)}"] if USE_AUG else []), *(["qkv"] if USE_QKV else []), *(["crd"] if USE_CRD else []), *(["logit"] if USE_LOGIT else []),
                  *([] if (USE_QKV or USE_CRD or USE_LOGIT) else ["ce"])])
print(f"dataset: {DATASET_ROOT} | thí nghiệm: {name} | RUN_TRAINING={RUN_TRAINING} | thiết bị: {device}")
summary, table = run_experiment(name)
print(summary)
table.round(4)